## Setup and load data

I load the cleaned trajectories from the Kaggle dataset and confirm the GPU. This is the single-task comparison notebook: I train a separate model per target and compare against the multi-task MT-TrajNet to answer RQ2.

In [1]:
import numpy as np
import pickle
import torch
import torch.nn as nn

device="cuda" if torch.cuda.is_available() else "cpu"
print("device:",device)
print("gpu:",torch.cuda.get_device_name(0) if device=="cuda" else "none")

DATA="/kaggle/input/datasets/arpitjoshua/mt-trajnet-thesis-data/kaggle_upload"

with open(DATA+"/assembled_trajectories.pkl","rb") as fh:
    data=pickle.load(fh)
X_traj=data["X_traj"]
y_arr=data["y_arr"]
groups=data["groups"]

print("loaded:",len(X_traj),"batches | y",y_arr.shape,"| codes",len(np.unique(groups)))
print("any NaN:",any(np.isnan(a).any() for a in X_traj))

device: cuda
gpu: Tesla T4
loaded: 1005 batches | y (1005, 4) | codes 25
any NaN: False


## Data preparation settings

I use the same trajectory preparation as the multi-task model: stride-2 downsampling and a length cap of 6000, so the single-task comparison is fair (same inputs, same resolution).

In [2]:
STRIDE=2
MAXLEN=6000
targets=["dissolution_av","tbl_av_hardness","tbl_rsd_weight","fct_tensile"]

def prep_batch(traj_list,mean,std,maxlen=MAXLEN):
    out=[]
    for a in traj_list:
        a=a[::STRIDE]
        a=(a-mean)/std
        if len(a)<maxlen:
            pad=np.zeros((maxlen-len(a),a.shape[1]),dtype="float32")
            a=np.vstack([a,pad])
        else:
            a=a[:maxlen]
        out.append(a)
    arr=np.array(out,dtype="float32")
    return torch.tensor(arr).transpose(1,2)

print("prep ready | stride",STRIDE,"maxlen",MAXLEN)

prep ready | stride 2 maxlen 6000


## Model classes: TCN encoder and single-task head

I use the same TCN encoder as the multi-task model, but with a single prediction head for one target. Each of the four targets gets its own separate model with its own encoder. This is the contrast to the shared-encoder multi-task model, and the comparison answers RQ2.

In [3]:
class TCNBlock(nn.Module):
    def __init__(self,in_ch,out_ch,dilation,kernel=3,dropout=0.1):
        super().__init__()
        pad=(kernel-1)*dilation
        self.conv1=nn.Conv1d(in_ch,out_ch,kernel,padding=pad,dilation=dilation)
        self.conv2=nn.Conv1d(out_ch,out_ch,kernel,padding=pad,dilation=dilation)
        self.relu=nn.ReLU()
        self.drop=nn.Dropout(dropout)
        self.pad=pad
        self.down=nn.Conv1d(in_ch,out_ch,1) if in_ch!=out_ch else None
    def forward(self,x):
        res=x if self.down is None else self.down(x)
        out=self.conv1(x)[:,:,:-self.pad]
        out=self.drop(self.relu(out))
        out=self.conv2(out)[:,:,:-self.pad]
        out=self.drop(self.relu(out))
        return self.relu(out+res)

class TCNEncoder(nn.Module):
    def __init__(self,in_ch=10,hidden=128,dropout=0.1):
        super().__init__()
        self.b1=TCNBlock(in_ch,hidden,1,dropout=dropout)
        self.b2=TCNBlock(hidden,hidden,2,dropout=dropout)
        self.b3=TCNBlock(hidden,hidden,4,dropout=dropout)
        self.b4=TCNBlock(hidden,hidden,8,dropout=dropout)
    def forward(self,x):
        x=self.b1(x);x=self.b2(x);x=self.b3(x);x=self.b4(x)
        return x.mean(dim=2)

class SingleTaskModel(nn.Module):
    def __init__(self,in_ch=10,hidden=128,dropout=0.1):
        super().__init__()
        self.encoder=TCNEncoder(in_ch,hidden,dropout)
        self.head=nn.Sequential(nn.Linear(hidden,64),nn.ReLU(),nn.Dropout(dropout),nn.Linear(64,1))
    def forward(self,x):
        z=self.encoder(x)
        return self.head(z).squeeze(-1)

m=SingleTaskModel().to(device)
test=torch.randn(2,10,500).to(device)
print("output shape:",m(test).shape)
print("parameters:",sum(p.numel() for p in m.parameters()))
print("model on:",next(m.parameters()).device)

output shape: torch.Size([2])
parameters: 358657
model on: cuda:0


## Single-task training function

For each target I train a separate model with its own encoder, using the same GroupKFold cross-validation and early stopping as the multi-task model. This keeps the comparison fair: the only difference is single-task versus shared-encoder multi-task. I report per-target RMSE with fold spread.

In [4]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
import warnings,time
warnings.filterwarnings("ignore")

def train_single_task(target_idx,n_splits=3,max_epochs=150,lr=5e-4,batch_size=16,patience=10):
    gkf=GroupKFold(n_splits=n_splits)
    fold_rmse=[]
    for fold,(trv,te) in enumerate(gkf.split(X_traj,y_arr[:,0],groups=groups)):
        trv_groups=groups[trv]; uniq=np.unique(trv_groups)
        rng=np.random.RandomState(42)
        val_codes=rng.choice(uniq,size=max(1,len(uniq)//5),replace=False)
        vm=np.isin(trv_groups,val_codes)
        tr=trv[~vm]; va=trv[vm]

        xmean=np.vstack([X_traj[i][::STRIDE] for i in tr]).mean(axis=0)
        xstd=np.vstack([X_traj[i][::STRIDE] for i in tr]).std(axis=0)+1e-6
        Xtr=prep_batch([X_traj[i] for i in tr],xmean,xstd)
        Xva=prep_batch([X_traj[i] for i in va],xmean,xstd)
        Xte=prep_batch([X_traj[i] for i in te],xmean,xstd)
        ymean=y_arr[tr,target_idx].mean(); ystd=y_arr[tr,target_idx].std()+1e-6
        ytr=torch.tensor((y_arr[tr,target_idx]-ymean)/ystd,dtype=torch.float32)
        yva=torch.tensor((y_arr[va,target_idx]-ymean)/ystd,dtype=torch.float32)
        yte=y_arr[te,target_idx]

        model=SingleTaskModel().to(device)
        opt=torch.optim.Adam(model.parameters(),lr=lr)
        lossf=nn.MSELoss()
        best_val=float("inf"); best_state=None; wait=0
        for ep in range(max_epochs):
            model.train()
            perm=torch.randperm(len(Xtr))
            for i in range(0,len(Xtr),batch_size):
                idx=perm[i:i+batch_size]
                xb=Xtr[idx].to(device); yb=ytr[idx].to(device)
                opt.zero_grad()
                loss=lossf(model(xb),yb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                opt.step()
            model.eval()
            with torch.no_grad():
                vp=[]
                for i in range(0,len(Xva),batch_size):
                    vp.append(model(Xva[i:i+batch_size].to(device)).cpu())
                vloss=lossf(torch.cat(vp),yva).item()
            if vloss<best_val-1e-4:
                best_val=vloss; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}; wait=0
            else:
                wait+=1
                if wait>=patience: break
        model.load_state_dict(best_state); model.eval()
        preds=[]
        with torch.no_grad():
            for i in range(0,len(Xte),batch_size):
                preds.append(model(Xte[i:i+batch_size].to(device)).cpu().numpy())
        pred=np.concatenate(preds)*ystd+ymean
        fold_rmse.append(np.sqrt(mean_squared_error(yte,pred)))
    return np.array(fold_rmse)

print("ready: single-task trainer")

ready: single-task trainer


## Train single-task models for all four targets

I train a separate model for each target and collect the per-target RMSE with fold spread. Comparing these against the multi-task MT-TrajNet answers RQ2: if multi-task matches or beats single-task, sharing the encoder helps; if single-task wins on a target, the shared encoder is hurting that target.

In [5]:
t0=time.time()
single_results={}
for j,tname in enumerate(targets):
    print(f"training single-task: {tname} ...")
    rmse=train_single_task(target_idx=j,n_splits=3,max_epochs=150,batch_size=16,patience=10)
    single_results[tname]={"mean":float(rmse.mean()),"std":float(rmse.std()),"folds":[float(r) for r in rmse]}
    print(f"  {tname}: {rmse.mean():.3f} (std {rmse.std():.3f})")
print()
print("all single-task models done, time:",round(time.time()-t0),"seconds")

training single-task: dissolution_av ...
  dissolution_av: 3.442 (std 0.320)
training single-task: tbl_av_hardness ...
  tbl_av_hardness: 7.794 (std 2.394)
training single-task: tbl_rsd_weight ...
  tbl_rsd_weight: 0.553 (std 0.118)
training single-task: fct_tensile ...
  fct_tensile: 0.318 (std 0.177)

all single-task models done, time: 1149 seconds


In [6]:
import json
rq2={
"comparison":"single-task vs multi-task (RQ2)",
"single_task":single_results,
"multi_task":{"dissolution_av":3.703,"tbl_av_hardness":7.598,"tbl_rsd_weight":0.554,"fct_tensile":0.349},
"finding":"mixed: multi-task helps hardness, single-task better on dissolution and tensile, weight RSD tied"
}
with open("/kaggle/working/rq2_single_vs_multi.json","w") as fh:
    json.dump(rq2,fh,indent=2)
print(json.dumps(rq2,indent=2))

{
  "comparison": "single-task vs multi-task (RQ2)",
  "single_task": {
    "dissolution_av": {
      "mean": 3.4416809922586524,
      "std": 0.3195355640299833,
      "folds": [
        3.0818010251464565,
        3.384937068517627,
        3.858304883111873
      ]
    },
    "tbl_av_hardness": {
      "mean": 7.793741616997416,
      "std": 2.394146430899396,
      "folds": [
        11.118427178004884,
        5.576642007600328,
        6.686155665387032
      ]
    },
    "tbl_rsd_weight": {
      "mean": 0.5526002685341009,
      "std": 0.11751067102567393,
      "folds": [
        0.4271558044942539,
        0.5209249288761848,
        0.7097200722318642
      ]
    },
    "fct_tensile": {
      "mean": 0.31830811943191756,
      "std": 0.1773948933349846,
      "folds": [
        0.1524038318608802,
        0.23828711267982233,
        0.5642334137550501
      ]
    }
  },
  "multi_task": {
    "dissolution_av": 3.703,
    "tbl_av_hardness": 7.598,
    "tbl_rsd_weight": 0.554,

## RQ2 significance: Nadeau-Bengio corrected test

For each target I apply the Nadeau-Bengio corrected paired test to the per-fold RMSE differences between single-task and multi-task. This adjusts for the overlap between cross-validation folds and tells me whether the differences are real or just noise. Only differences that pass this test should be claimed.

In [7]:
from scipy import stats

multi={
"dissolution_av":[3.793,3.311,4.006],
"tbl_av_hardness":[10.992,5.683,6.121],
"tbl_rsd_weight":[0.429,0.535,0.697],
"fct_tensile":[0.213,0.241,0.595],
}
single={t:single_results[t]["folds"] for t in targets}

n_splits=3
n_test_over_train=0.5

print(f"{'target':<18}{'single':<9}{'multi':<9}{'diff':<9}{'p-value':<10}{'significant'}")
for t in targets:
    d=np.array(single[t])-np.array(multi[t])   
    n=len(d)
    mean_d=d.mean(); var_d=d.var(ddof=1)
    corrected_var=var_d*(1/n + n_test_over_train)
    t_stat=mean_d/np.sqrt(corrected_var) if corrected_var>0 else 0
    p=2*(1-stats.t.cdf(abs(t_stat),n-1))
    sig="yes" if p<0.05 else "no"
    print(f"{t:<18}{np.mean(single[t]):<9.3f}{np.mean(multi[t]):<9.3f}{mean_d:<9.3f}{p:<10.3f}{sig}")

print("\nNegative diff = single-task better. 'significant' = difference is real,")
print("not just noise, under the corrected test.")

target            single   multi    diff     p-value   significant
dissolution_av    3.442    3.703    -0.262   0.552     no
tbl_av_hardness   7.794    7.599    0.195    0.595     no
tbl_rsd_weight    0.553    0.554    -0.001   0.939     no
fct_tensile       0.318    0.350    -0.031   0.357     no

Negative diff = single-task better. 'significant' = difference is real,
not just noise, under the corrected test.


In [8]:
import json
from scipy import stats
multi_folds={
"dissolution_av":[3.793,3.311,4.006],
"tbl_av_hardness":[10.992,5.683,6.121],
"tbl_rsd_weight":[0.429,0.535,0.697],
"fct_tensile":[0.213,0.241,0.595],
}
multi_mean={"dissolution_av":3.703,"tbl_av_hardness":7.598,"tbl_rsd_weight":0.554,"fct_tensile":0.349}

def nb_pvalue(single_folds,multi_folds):
    d=np.array(single_folds)-np.array(multi_folds)
    n=len(d); mean_d=d.mean(); var_d=d.var(ddof=1)
    cv=var_d*(1/n+0.5)
    tstat=mean_d/np.sqrt(cv) if cv>0 else 0
    return float(2*(1-stats.t.cdf(abs(tstat),n-1)))

rq2_sig={"test":"Nadeau-Bengio corrected, single-task vs multi-task per target","results":{}}
for t in targets:
    p=nb_pvalue(single_results[t]["folds"],multi_folds[t])
    rq2_sig["results"][t]={
        "single":round(single_results[t]["mean"],3),
        "multi":multi_mean[t],
        "p":round(p,3),
        "significant":bool(p<0.05)
    }
rq2_sig["conclusion"]="no significant difference on any target; multi-task preferred on efficiency grounds; shared encoder does not harm any target"

with open("/kaggle/working/rq2_significance.json","w") as fh:
    json.dump(rq2_sig,fh,indent=2)
print(json.dumps(rq2_sig,indent=2))

{
  "test": "Nadeau-Bengio corrected, single-task vs multi-task per target",
  "results": {
    "dissolution_av": {
      "single": 3.442,
      "multi": 3.703,
      "p": 0.552,
      "significant": false
    },
    "tbl_av_hardness": {
      "single": 7.794,
      "multi": 7.598,
      "p": 0.595,
      "significant": false
    },
    "tbl_rsd_weight": {
      "single": 0.553,
      "multi": 0.554,
      "p": 0.939,
      "significant": false
    },
    "fct_tensile": {
      "single": 0.318,
      "multi": 0.349,
      "p": 0.357,
      "significant": false
    }
  },
  "conclusion": "no significant difference on any target; multi-task preferred on efficiency grounds; shared encoder does not harm any target"
}


## Final summary: RQ2 comparison

A clean summary of the single-task versus multi-task comparison across all four targets, with the Nadeau-Bengio significance result. This is the answer to RQ2.

In [9]:
import pandas as pd

summary=pd.DataFrame({
"single_task":{t:round(single_results[t]["mean"],3) for t in targets},
"multi_task":{t:multi_mean[t] for t in targets},
"p_value":{t:rq2_sig["results"][t]["p"] for t in targets},
})
summary["significant"]=summary["p_value"].apply(lambda p:"yes" if p<0.05 else "no")
summary["better"]=summary.apply(lambda r:"single" if r["single_task"]<r["multi_task"] else "multi",axis=1)

print("RQ2: SINGLE-TASK vs MULTI-TASK (RMSE, lower better)")
print(summary)
print("\nConclusion: no significant difference on any target.")
print("Shared encoder does not harm any target; multi-task preferred on efficiency.")

summary.to_csv("/kaggle/working/rq2_summary_table.csv")
print("\nsaved rq2_summary_table.csv")

RQ2: SINGLE-TASK vs MULTI-TASK (RMSE, lower better)
                 single_task  multi_task  p_value significant  better
dissolution_av         3.442       3.703    0.552          no  single
tbl_av_hardness        7.794       7.598    0.595          no   multi
tbl_rsd_weight         0.553       0.554    0.939          no  single
fct_tensile            0.318       0.349    0.357          no  single

Conclusion: no significant difference on any target.
Shared encoder does not harm any target; multi-task preferred on efficiency.

saved rq2_summary_table.csv
